In [67]:

from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("retail_data.csv")
df

,Unnamed: 0,Customer_ID,Age,Annual_Income,City,Mixed_ID,Transaction_Date,Spending_Score,Annual_Income_Filled,Year,Month,Day
0,0,1000,39,NaN,New York,ID_0,2023-01-01 00:00:00,500,0.000000,2023,1,1
1,1,1001,33,87501.712418,NaN,10.0,2023-01-02 00:00:00,600,87501.712418,2023,1,2
2,2,1002,41,NaN,New York,ID_2,2023-01-03 04:00:00,-150,0.000000,2023,1,3
3,3,1003,50,80614.377811,New York,30.0,2023-01-04 00:00:00,800,80614.377811,2023,1,4
4,4,1004,32,42274.617116,New York,ID_4,2023-01-05 08:00:00,1000,42274.617116,2023,1,5
...,...,...,...,...,...,...,...,...,...,...,...,...
195,195,1195,38,26324.549797,New York,1950.0,2023-07-31 00:00:00,3,26324.549797,2023,7,31
196,196,1196,26,26251.311600,Tokyo,ID_196,2023-08-01 00:00:00,32,26251.311600,2023,8,1
197,197,1197,36,NaN,Dubai,1970.0,2023-08-02 00:00:00,10,0.000000,2023,8,2
198,198,1198,35,88529.885880,New York,ID_198,2023-08-03 12:00:00,74,88529.885880,2023,8,3


In [71]:
df['Annual_Income_Filled'] = df['Annual_Income'].fillna(0)
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'], format='mixed')
df['Year'] = df['Transaction_Date'].dt.year
df['Month'] = df['Transaction_Date'].dt.month
df['Day'] = df['Transaction_Date'].dt.day
data = df[['Age', 'Annual_Income_Filled', 'Year', 'Month', 'Day', 'Spending_Score']].dropna()
X = data.drop('Spending_Score', axis=1)
y = data['Spending_Score']



import pandas as pd
import statsmodels.api as sm

# Prepare data (dropping NaNs for regression)
data = df[['Age', 'Annual_Income_Filled', 'Year', 'Month', 'Day', 'Spending_Score']].dropna()
X = data.drop('Spending_Score', axis=1)
y = data['Spending_Score']

def forward_selection(data, target, significance_level=0.05):
    initial_features = data.columns.tolist()
    best_features = []
    while len(initial_features) > 0:
        remaining_features = list(set(initial_features) - set(best_features))
        new_pval = pd.Series(index=remaining_features, dtype='float64')
        for new_column in remaining_features:
            model = sm.OLS(target, sm.add_constant(data[best_features + [new_column]])).fit()
            new_pval[new_column] = model.pvalues[new_column]
        min_p_value = new_pval.min()
        if min_p_value < significance_level:
            best_features.append(new_pval.idxmin())
        else:
            break
    return best_features

selected_features_fwd = forward_selection(X, y)
print(f"Forward Selection chosen features: {selected_features_fwd}")

In [72]:
import pandas as pd
import statsmodels.api as sm

# Prepare data (dropping NaNs for regression)
data = df[['Age', 'Annual_Income_Filled', 'Year', 'Month', 'Day', 'Spending_Score']].dropna()
X = data.drop('Spending_Score', axis=1)
y = data['Spending_Score']

def forward_selection(data, target, significance_level=0.05):
    initial_features = data.columns.tolist()
    best_features = []
    while len(initial_features) > 0:
        remaining_features = list(set(initial_features) - set(best_features))
        new_pval = pd.Series(index=remaining_features, dtype='float64')
        for new_column in remaining_features:
            model = sm.OLS(target, sm.add_constant(data[best_features + [new_column]])).fit()
            new_pval[new_column] = model.pvalues[new_column]
        min_p_value = new_pval.min()
        if min_p_value < significance_level:
            best_features.append(new_pval.idxmin())
        else:
            break
    return best_features

selected_features_fwd = forward_selection(X, y)
print(f"Forward Selection chosen features: {selected_features_fwd}")

Forward Selection chosen features: ['Year', 'Month', 'Day']


In [73]:
df.to_csv("retail_data.csv")

In [74]:
df

,Unnamed: 0,Customer_ID,Age,Annual_Income,City,Mixed_ID,Transaction_Date,Spending_Score,Annual_Income_Filled,Year,Month,Day
0,0,1000,39,NaN,New York,ID_0,2023-01-01 00:00:00,500,0.000000,2023,1,1
1,1,1001,33,87501.712418,NaN,10.0,2023-01-02 00:00:00,600,87501.712418,2023,1,2
2,2,1002,41,NaN,New York,ID_2,2023-01-03 04:00:00,-150,0.000000,2023,1,3
3,3,1003,50,80614.377811,New York,30.0,2023-01-04 00:00:00,800,80614.377811,2023,1,4
4,4,1004,32,42274.617116,New York,ID_4,2023-01-05 08:00:00,1000,42274.617116,2023,1,5
...,...,...,...,...,...,...,...,...,...,...,...,...
195,195,1195,38,26324.549797,New York,1950.0,2023-07-31 00:00:00,3,26324.549797,2023,7,31
196,196,1196,26,26251.311600,Tokyo,ID_196,2023-08-01 00:00:00,32,26251.311600,2023,8,1
197,197,1197,36,NaN,Dubai,1970.0,2023-08-02 00:00:00,10,0.000000,2023,8,2
198,198,1198,35,88529.885880,New York,ID_198,2023-08-03 12:00:00,74,88529.885880,2023,8,3


In [76]:
def backward_elimination(data, target, significance_level = 0.05):
    features = data.columns.tolist()
    while len(features) > 0:
        features_with_const = sm.add_constant(data[features])
        p_values = sm.OLS(target, features_with_const).fit().pvalues[1:]
        max_pvalues = p_values.max()
        if max_pvalues > significance_level:
            excluded_feature = p_values.idxmax()
            features.remove(excluded_feature)
            print(f"Removing {excluded_feature} with p-values {max_pvalues}")
        else:
            break
    return features

selected_backf = backward_elimination(X, y)
print("here are the selected features", selected_backf)

Removing Annual_Income_Filled with p-values 0.5589853112030003
here are the selected features ['Age', 'Year', 'Month', 'Day']
